In [ ]:
"""
Visualize ILI Component Detection dataset and training pipeline.

Shows: (1) raw + augmented stacked per sample, (2) class distribution.
"""
import copy
import random
import numpy as np
import matplotlib.pyplot as plt
import cv2
from detectron2.data import DatasetCatalog, MetadataCatalog
from detectron2.utils.visualizer import Visualizer

from utils import load_config
from Dataset import register_detection_datasets

In [ ]:
config = load_config('config.yaml')
random.seed(config['random_state'])

# Register datasets and get training samples
n_total, n_train, n_val, num_classes, class_names = register_detection_datasets(config)
print(f"Datasets: total={n_total}, train={n_train}, val={n_val}")
print(f"Classes ({num_classes}): {class_names}")

dataset_dicts = DatasetCatalog.get("data_detection_train")
metadata = MetadataCatalog.get("data_detection_train")

In [ ]:
# === Single visualization cell: class distribution + original vs augmented ===
from collections import Counter

num_samples = 6
samples = random.sample(dataset_dicts, min(num_samples, len(dataset_dicts)))
cat_counts = Counter()

for d in dataset_dicts:
    for ann in d.get("annotations", []):
        cid = ann["category_id"]
        cat_counts[metadata.thing_classes[cid]] += 1
print("Instances per class:")
for name, count in cat_counts.most_common():
    print(f"  {name}: {count}")

fig1, ax_bar = plt.subplots(figsize=(10, 4))
names, counts = zip(*cat_counts.most_common()) if cat_counts else ([], [])
bars = ax_bar.bar(names, counts, color=plt.cm.Set3(np.linspace(0, 1, max(1, len(names)))))
ax_bar.set_ylabel("Count")
ax_bar.set_title("Class distribution")
ax_bar.tick_params(axis="x", rotation=45)
for bar, c in zip(bars, counts):
    ax_bar.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 2, str(c), ha="center", fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
mapper = None
try:
    from trainer import TrainerModule
    from data import ILIDatasetMapper
    module = TrainerModule("config.yaml")
    mapper = ILIDatasetMapper(
        module.cfg, is_train=True,
        num_tracks=getattr(module.cfg, "NUM_TRACKS", 22),
        max_track_shift=getattr(module.cfg, "MAX_TRACK_SHIFT", 15),
        track_shift_prob=getattr(module.cfg, "TRACK_SHIFT_PROB", 0.5),
        circular_roll_prob=getattr(module.cfg, "CIRCULAR_ROLL_PROB", 0.3),
        split_wrapped_boxes=getattr(module.cfg, "SPLIT_WRAPPED_BOXES", True),
        force_augment_for_viz=True,
    )
except Exception as e:
    print(f"Mapper skipped: {e}")

n_cols = min(3, num_samples)
n_rows = int(np.ceil(num_samples / n_cols))
fig2, axes = plt.subplots(n_rows * 2, n_cols, figsize=(5 * n_cols, 5 * n_rows * 2))
axes = np.array(axes).reshape(n_rows * 2, n_cols)
for i, d in enumerate(samples):
    col, row_base = i % n_cols, 2 * (i // n_cols)
    ax_orig, ax_aug = axes[row_base, col], axes[row_base + 1, col]
    # Original
    img_orig = cv2.imread(d["file_name"])[:, :, ::-1]
    v_orig = Visualizer(img_orig, metadata=metadata, scale=0.8)
    ax_orig.imshow(v_orig.draw_dataset_dict(d).get_image())
    ax_orig.set_title("Original" if col == 0 else "")
    ax_orig.axis("off")
    # Augmented
    if mapper is not None:
        try:
            d_aug = mapper(copy.deepcopy(d))
            img_aug = d_aug["image"].permute(1, 2, 0).numpy()
            # Mapper outputs 0-255; use raw values to preserve grayscale tones (avoid B&W look)
            if img_aug.dtype in (np.float32, np.float64):
                img_aug = np.clip(img_aug * 255 if img_aug.max() <= 1 else img_aug, 0, 255).astype(np.uint8)
            else:
                img_aug = np.clip(img_aug.astype(np.float32), 0, 255).astype(np.uint8)
            img_aug = img_aug[:, :, ::-1]  # BGR -> RGB
            v_aug = Visualizer(img_aug, metadata=metadata, scale=0.8)
            ax_aug.imshow(v_aug.draw_instance_predictions(d_aug["instances"].to("cpu")).get_image())
        except Exception as e:
            ax_aug.imshow(img_orig)
            ax_aug.set_title(str(e)[:30] if col != 0 else "Augmented (error)", fontsize=8)
        else:
            ax_aug.set_title("Augmented" if col == 0 else "")
    else:
        ax_aug.imshow(img_orig)
        ax_aug.set_title("(no mapper)" if col != 0 else "Augmented")
    ax_aug.axis("off")
for j in range(num_samples, n_cols * n_rows):
    r, c = 2 * (j // n_cols), j % n_cols
    axes[r, c].axis("off")
    axes[r + 1, c].axis("off")
plt.suptitle("Original (top) vs Augmented (bottom)", fontsize=14)
plt.tight_layout()
plt.show()